In [1]:
import numpy as np
import pandas as pd
import faiss
from sklearn.decomposition import PCA

K = 5
PCA_DIMS = [512, 256, 128]

dataset = pd.read_parquet("data/rag_qa_ptbr_chunk_embeddings.parquet")
embedded_questions_df = pd.read_parquet("data/encoded_questions.parquet")


In [2]:
# Document vectors
index_df = dataset[["document_id", "embedding"]].copy()
index_df = index_df[index_df["embedding"].notna()].reset_index(drop=True)

doc_ids = index_df["document_id"].astype(str).to_numpy()
X_full = np.vstack(index_df["embedding"].values).astype("float32")

# Query vectors
queries_df = embedded_questions_df[["document_id", "question_embedding"]].copy()
queries_df = queries_df[queries_df["question_embedding"].notna()].reset_index(drop=True)

gold_doc_ids = queries_df["document_id"].astype(str).to_numpy()
Q_full = np.vstack(queries_df["question_embedding"].values).astype("float32")

print("Document vectors:", X_full.shape)
print("Question vectors:", Q_full.shape)

Document vectors: (1420215, 1024)
Question vectors: (17138, 1024)


In [3]:
def evaluate_retrieval(indices, doc_ids, gold_doc_ids, k=5):
    hits = []
    reciprocal_ranks = []

    for q_idx in range(len(gold_doc_ids)):
        gold_doc_id = gold_doc_ids[q_idx]

        retrieved_doc_ids = [
            doc_ids[idx]
            for idx in indices[q_idx]
            if idx != -1
        ]

        if gold_doc_id in retrieved_doc_ids:
            rank = retrieved_doc_ids.index(gold_doc_id) + 1
            hits.append(1)
            reciprocal_ranks.append(1.0 / rank)
        else:
            hits.append(0)
            reciprocal_ranks.append(0.0)

    return {
        "hit_rate@5": float(np.mean(hits)),
        "mrr@5": float(np.mean(reciprocal_ranks)),
    }


def vector_size_mb(num_vectors, dim, bytes_per_value):
    return num_vectors * dim * bytes_per_value / 1024**2

In [4]:
gpu_id = 0
res = faiss.StandardGpuResources()

def search_faiss_gpu(X, Q, k=5, use_float16=False):
    X = np.ascontiguousarray(X.astype("float32"))
    Q = np.ascontiguousarray(Q.astype("float32"))

    faiss.normalize_L2(X)
    faiss.normalize_L2(Q)

    dim = X.shape[1]

    config = faiss.GpuIndexFlatConfig()
    config.device = gpu_id
    config.useFloat16 = use_float16

    index = faiss.GpuIndexFlatIP(res, dim, config)
    index.add(X)

    scores, indices = index.search(Q, k)

    return scores, indices

In [5]:
results = []

full_dim = X_full.shape[1]
full_size_fp32_mb = vector_size_mb(len(X_full), full_dim, 4)

for precision_name, use_float16, bytes_per_value in [
    ("float32", False, 4),
    ("float16", True, 2),
]:
    print(f"\nSearching full dim, {precision_name}...")

    scores, indices = search_faiss_gpu(
        X_full,
        Q_full,
        k=K,
        use_float16=use_float16,
    )

    metrics = evaluate_retrieval(
        indices=indices,
        doc_ids=doc_ids,
        gold_doc_ids=gold_doc_ids,
        k=K,
    )

    size_mb = vector_size_mb(len(X_full), full_dim, bytes_per_value)

    results.append({
        "method": f"full_{precision_name}",
        "dim": full_dim,
        "precision": precision_name,
        "vector_size_mb": size_mb,
        "compression_vs_full_fp32": full_size_fp32_mb / size_mb,
        "explained_variance": 1.0,
        "hit_rate@5": metrics["hit_rate@5"],
        "mrr@5": metrics["mrr@5"],
    })

    print(f"Hit Rate@5: {metrics['hit_rate@5']:.4f}")
    print(f"MRR@5:      {metrics['mrr@5']:.4f}")
    print(f"Size MB:    {size_mb:.2f}")


for target_dim in PCA_DIMS:
    if target_dim >= full_dim:
        print(f"Skipping PCA {target_dim}: target_dim >= full_dim")
        continue

    print(f"\nFitting PCA to {target_dim} dimensions...")

    pca = PCA(
        n_components=target_dim,
        svd_solver="randomized",
        random_state=42,
    )

    X_pca = pca.fit_transform(X_full).astype("float32")
    Q_pca = pca.transform(Q_full).astype("float32")

    explained = float(pca.explained_variance_ratio_.sum())

    print(f"Explained variance: {explained:.4f}")

    for precision_name, use_float16, bytes_per_value in [
        ("float32", False, 4),
        ("float16", True, 2),
    ]:
        print(f"Searching PCA {target_dim}, {precision_name}...")

        scores, indices = search_faiss_gpu(
            X_pca,
            Q_pca,
            k=K,
            use_float16=use_float16,
        )

        metrics = evaluate_retrieval(
            indices=indices,
            doc_ids=doc_ids,
            gold_doc_ids=gold_doc_ids,
            k=K,
        )

        size_mb = vector_size_mb(len(X_pca), target_dim, bytes_per_value)

        results.append({
            "index": f"pca_{target_dim}_{precision_name}",
            "approx_vector_storage_mb": size_mb,
            "compression_vs_fp32": full_size_fp32_mb / size_mb,
            # "explained_variance": explained,
            "hit_rate@5": metrics["hit_rate@5"],
            "mrr@5": metrics["mrr@5"],
        })

        print(f"Hit Rate@5: {metrics['hit_rate@5']:.4f}")
        print(f"MRR@5:      {metrics['mrr@5']:.4f}")
        print(f"Size MB:    {size_mb:.2f}")


Searching full dim, float32...
Hit Rate@5: 0.8278
MRR@5:      0.7457
Size MB:    5547.71

Searching full dim, float16...
Hit Rate@5: 0.8278
MRR@5:      0.7457
Size MB:    2773.86

Fitting PCA to 512 dimensions...
Explained variance: 0.9410
Searching PCA 512, float32...
Hit Rate@5: 0.8189
MRR@5:      0.7353
Size MB:    2773.86
Searching PCA 512, float16...
Hit Rate@5: 0.8189
MRR@5:      0.7353
Size MB:    1386.93

Fitting PCA to 256 dimensions...
Explained variance: 0.7983
Searching PCA 256, float32...
Hit Rate@5: 0.7990
MRR@5:      0.7129
Size MB:    1386.93
Searching PCA 256, float16...
Hit Rate@5: 0.7990
MRR@5:      0.7129
Size MB:    693.46

Fitting PCA to 128 dimensions...
Explained variance: 0.6115
Searching PCA 128, float32...
Hit Rate@5: 0.7542
MRR@5:      0.6603
Size MB:    693.46
Searching PCA 128, float16...
Hit Rate@5: 0.7542
MRR@5:      0.6603
Size MB:    346.73


In [6]:
summary_df = pd.DataFrame(results)

summary_df = summary_df[
    [
        "index",
        "approx_vector_storage_mb",
        "compression_vs_fp32",
        "hit_rate@5",
        "mrr@5",
    ]
].reset_index(drop=True)


In [7]:
import pandas as pd

csv_path = "data/compression_results.csv"

results_df = pd.read_csv(csv_path)

new_rows_df = summary_df[results_df.columns]

updated_df = pd.concat(
    [results_df, new_rows_df],
    ignore_index=True
)

updated_df.to_csv("data/updated_"+csv_path, index=False)